# LOADING LIBRARIES

In [14]:
import pandas as pd
import numpy as np
import os

 # LOADING THE FILES 

In [15]:
import glob
path = r"C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE"
files = glob.glob(path + r"\*.csv")
print(type(files))   # should be list
print(len(files))  
for f in files:
    print(f)

<class 'list'>
8
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Morning.pcap_ISCX.csv
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Wednesday-workingHours.pcap_ISCX.csv


 # CHECKING CONSISTENCY OF SCHEMA FOR ALL THE CSV FILES

In [16]:
columns_map = {}

for f in files:
    cols = pd.read_csv(f, nrows=0).columns.str.strip()
    columns_map[f] = list(cols)

# Use first file as reference
base_file = files[0]
base_cols = columns_map[base_file]

print(f"Reference file:\n{base_file}")
print(f"Total columns: {len(base_cols)}")

all_good = True

for f in files:
    cols = columns_map[f]
    
    if cols != base_cols:
        all_good = False
        print(f"\n❌ Issue in: {f}")
        
        missing = set(base_cols) - set(cols)
        extra   = set(cols) - set(base_cols)
        
        if missing:
            print("  Missing columns:", missing)
        if extra:
            print("  Extra columns:", extra)
        
        if set(cols) == set(base_cols):
            print("  ⚠️ Same columns but DIFFERENT ORDER")
    else:
        print(f"✅ {f} matches")

if all_good:
    print("\n✅ All files consistent. Safe to proceed.")
else:
    print("\n⚠️ Fix issues before merging.")

Reference file:
C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Total columns: 79
✅ C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv matches
✅ C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv matches
✅ C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Morning.pcap_ISCX.csv matches
✅ C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv matches
✅ C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv matches
✅ C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv matches
✅ C:\Users\lenovo\Downloads\MachineLearningCSV\MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv matches
✅ C:\Us

# MERGING THE FILES

In [17]:
df_list = []
for f in files:
    temp = pd.read_csv(f)
    # safety: strip column names again
    temp.columns = temp.columns.str.strip()
    # enforce same order (critical even if already matching)
    temp = temp[base_cols]
    df_list.append(temp)
df = pd.concat(df_list, ignore_index=True)
print("Merged shape:", df.shape)

Merged shape: (2830743, 79)


In [18]:
print(df.head())
print("Columns:", len(df.columns))
print("Label exists:", 'Label' in df.columns)

   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0             54865              3                  2                       0   
1             55054            109                  1                       1   
2             55055             52                  1                       1   
3             46236             34                  1                       1   
4             54863              3                  2                       0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                           12                            0   
1                            6                            6   
2                            6                            6   
3                            6                            6   
4                           12                            0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                      6                      6            

# CHECKING FOR MISSING VALUES AND INFINITE VALUES 

In [19]:
numeric_df = df.select_dtypes(include=[np.number])
print("NaN values:", df.isnull().sum().sum())
print("Inf values:", np.isinf(numeric_df).sum().sum())

NaN values: 1358
Inf values: 4376


# CONVERTING THE INFINITE VALUES INTO NAN AND REMOVING ALL THE NAN VALUES AT ONCE!

In [20]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print("NaN after replacing Inf:", df.isnull().sum().sum())

NaN after replacing Inf: 5734


# CALCULATING NO OF ROWS AFFECTED 

In [21]:
# No of rows affected by the nan values
rows_with_nan = df.isnull().any(axis=1).sum()
print("Rows containing NaN:", rows_with_nan)
rows_before = df.shape[0]

Rows containing NaN: 2867


# DELETING THE NAN ROWS 

In [22]:
df.dropna(inplace=True)

# CALCULATING NEW ROWS AFTER DELETION

In [23]:
rows_after_nan = df.shape[0]
print("Rows removed (NaN/Inf):", rows_before - rows_after_nan)

Rows removed (NaN/Inf): 2867


# COUNTING EXACT DUPLICATES

In [24]:
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)


Duplicate rows: 307078


# DELETING  EXACT DUPULICATES 

In [25]:
df.drop_duplicates(inplace=True)

# CALCULATING NO OF ROWS NOW AFFECTED AFTER REMOVING DUPLICATES TOO!

In [26]:
rows_after_dup = df.shape[0]
print("Rows removed (duplicates):", rows_after_nan - rows_after_dup)

Rows removed (duplicates): 307078


# FIXING LABEL FORMATTING 

In [27]:
df['Label'] = df['Label'].str.strip()

# CALCULATING EXACT NO OF CONFLICTING DUPLICATES

In [28]:
conflicts = df.groupby(list(df.columns[:-1]))['Label'].nunique()
conflicts = conflicts[conflicts > 1]
print("Conflicting duplicates:", len(conflicts))

Conflicting duplicates: 697


# DELETING THE CONFLICTING DUPLICATES 

In [29]:
conflict_mask = df.groupby(list(df.columns[:-1]))['Label'].transform('nunique') > 1
df = df[~conflict_mask]
df.drop_duplicates(inplace=True)

# FINAL VERIFICATION

In [30]:
numeric_df = df.select_dtypes(include='number')
print("\nFinal shape:", df.shape)
print("Remaining NaN:", df.isnull().sum().sum())
print("Remaining Inf:", np.isinf(numeric_df).sum().sum())
print("Remaining duplicates:", df.duplicated().sum())
conflicts = df.groupby(list(df.columns[:-1]))['Label'].nunique()
conflicts = conflicts[conflicts > 1]
print("Remaining conflicts:", len(conflicts))


Final shape: (2519404, 79)
Remaining NaN: 0
Remaining Inf: 0
Remaining duplicates: 0
Remaining conflicts: 0


# CALCULATING HOW MANY ROWS WERE DELETED IN  TOTAL AFTER WHOLE CLEANUP

In [31]:
original_rows = 2830743   # from merge step
current_rows = df.shape[0]
print("Current rows:", current_rows)
print("Total rows removed:", original_rows - current_rows)

Current rows: 2519404
Total rows removed: 311339


# SAVING CLEANED DATASET

In [32]:
df.to_csv("cleaned_data.csv", index=False)